In [ ]:
# [NB04-C01] SETUP
# What: install the pinned pm4py version, import the libraries, create the
#       figures folder.
# Why: same environment as NB01-NB03 so all the notebooks agree.
# Expected: no errors.

%pip install pm4py==2.7.23.1 -q

import os                                          # filesystem helpers
import pandas as pd                                # dataframes and CSV handling
import numpy as np                                 # numeric helpers
import matplotlib.pyplot as plt                    # plots
import seaborn as sns                              # nicer statistical plots
import pm4py                                       # process mining library of the course
from pm4py.util import constants                   # to silence the replay progress bars
from sklearn.cluster import AgglomerativeClustering    # clustering of the binary signatures
from sklearn.metrics import silhouette_score           # to choose the number of clusters
from sklearn.metrics.pairwise import pairwise_distances  # Jaccard distance between signatures
from scipy.stats import kruskal, mannwhitneyu      # non-parametric tests (chosen in NB01)

# Silence the replay progress bars: they would flood the notebook output
constants.SHOW_PROGRESS_BAR = False

# Fixed seed: every result of this notebook is reproducible
RANDOM_STATE = 42

# Create the folder where the report figures will be saved (safe if it exists)
os.makedirs('figures', exist_ok=True)

In [ ]:
# [NB04-C02] LOAD THE PREPARED FILES
# What: load the raw view (with the quality flags), the abstract view and the
#       case table produced by NB01/NB02.
# Why: the clinical patterns are computed on the RAW view, because that is
#      where every clinical recording lives (the abstract view collapses
#      same-instant recordings and would lose measurements). The abstract
#      view is used only for the control-flow analyses. NO row is dropped.
# Expected: 25,115 raw rows, 16,826 abstract rows, 1,820 stays.

# Paths of the prepared files (on Colab: upload them and update the paths)
raw_path = 'prepared_event_log.csv'
abstract_path = 'abstract_event_log.csv'
cases_path = 'case_table.csv'

# Defensive check: stop with a clear message if a file is missing
for p in [raw_path, abstract_path, cases_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(p + ' not found - run NB01 and NB02 first.')

# Load the raw event log with the quality flags built in NB01
raw_log = pd.read_csv(raw_path, parse_dates=['time:timestamp'])
raw_log['case:concept:name'] = raw_log['case:concept:name'].astype(str)

# Load the abstract control-flow view
abstract_log = pd.read_csv(abstract_path, parse_dates=['time:timestamp'])
abstract_log['case:concept:name'] = abstract_log['case:concept:name'].astype(str)

# Load the case table (one row per stay)
case_table = pd.read_csv(cases_path, index_col=0, parse_dates=['start_time', 'end_time'])
case_table.index = case_table.index.astype(str)

print('Raw view rows: {} (expected 25,115)'.format(len(raw_log)))
print('Abstract view rows: {} (expected 16,826)'.format(len(abstract_log)))
print('Stays: {} (expected 1,820)'.format(len(case_table)))

In [ ]:
# [NB04-C03] WHEN IS EACH ATTRIBUTE KNOWN?
# What: check, for each attribute used by the patterns, on WHICH activity it
#       is actually recorded.
# Why: this decides two things. First, which patterns describe the CONTEXT of
#      a stay and which ones describe its OUTCOME: a property recorded at
#      discharge cannot be a context feature for prediction (it would be
#      target leakage, the lesson of Case6). Second, which patterns NB05 can
#      reuse to predict from an early prefix. Verified on the data, not
#      assumed.
# Expected: acuity only at triage, disposition and diagnosis only at
#      discharge, vitals at triage and at vital sign checks.

# For every relevant attribute, on which activities does it carry a value?
for column in ['arrival_transport', 'acuity', 'temperature', 'pain', 'rhythm', 'disposition', 'diagnosis_code']:
    activities = raw_log[raw_log[column].notna()]['concept:name'].value_counts()
    print('{:18s} recorded on: {}'.format(column, dict(activities)))

# Interpretation: arrival_transport is known at arrival, acuity and the first
# vitals at triage, further vitals during the stay, while disposition and
# diagnosis exist ONLY on the discharge event. Patterns built on the latter
# describe the outcome of a stay, not its context: they are used here for
# description, and are excluded from the predictive features of NB05.

In [ ]:
# [NB04-C04] THE PATTERN DICTIONARY (VERSIONED)
# What: define every candidate property with its definition, its rationale,
#       its data source and the moment it becomes known.
# Why: the additional question asks to encode each stay with a binary vector
#      over a set of properties. The set is the method: it must be explicit,
#      auditable and versioned, otherwise the encoding cannot be discussed
#      or reproduced. Clinical thresholds come from the Appendix of the case
#      study and are applied ONLY to plausible values (the *_abnormal flags
#      built in NB01-C11 already do this).
# Expected: a readable dictionary, exported for the report.

# The dictionary. 'source' says where the property comes from:
#   Appendix   = clinical threshold given by the case study
#   scenario   = case attribute described by the case study
#   discovered = property this project found in its own analysis
#   process    = structural property of the control flow
pattern_dictionary = pd.DataFrame([
    {'pattern': 'acuity_1_3', 'source': 'Appendix', 'known_from': 'triage',
     'definition': 'acuity in {1,2,3}',
     'rationale': 'the Appendix marks acuity 1-3 as the abnormal (urgent) range'},
    {'pattern': 'acuity_1_2', 'source': 'refinement', 'known_from': 'triage',
     'definition': 'acuity in {1,2}',
     'rationale': 'acuity_1_3 covers 88.7% of stays and barely discriminates; 1-2 isolates the very urgent'},
    {'pattern': 'ambulance_arrival', 'source': 'scenario', 'known_from': 'arrival',
     'definition': 'arrival_transport = AMBULANCE',
     'rationale': 'pre-hospital urgency: a different entry route into the ED'},
    {'pattern': 'pain_any_gt_0', 'source': 'Appendix', 'known_from': 'triage',
     'definition': 'max plausible pain > 0',
     'rationale': 'the Appendix marks any pain above 0 as abnormal; drives analgesia workload'},
    {'pattern': 'tachycardia_any_gt_100', 'source': 'Appendix', 'known_from': 'triage',
     'definition': 'max plausible heart rate > 100 bpm',
     'rationale': 'Appendix threshold for tachycardia'},
    {'pattern': 'vitals_before_triage', 'source': 'discovered', 'known_from': 'triage',
     'definition': 'a Vital sign check occurs before the Triage event',
     'rationale': 'found by the conformance analysis of NB03-C19: the initial assessment is a composite step'},
    # --- extended set: documented, used for the fragmentation analysis and clustering
    {'pattern': 'fever_max_ge_100F', 'source': 'Appendix', 'known_from': 'during stay',
     'definition': 'max plausible temperature >= 100 F',
     'rationale': 'Appendix threshold for fever'},
    {'pattern': 'tachypnea_any_gt_20', 'source': 'Appendix', 'known_from': 'during stay',
     'definition': 'max plausible respiratory rate > 20 breaths/min',
     'rationale': 'Appendix threshold for tachypnea'},
    {'pattern': 'hypoxemia_any_lt_90', 'source': 'Appendix', 'known_from': 'during stay',
     'definition': 'min plausible oxygen saturation < 90 %',
     'rationale': 'Appendix threshold for hypoxemia'},
    {'pattern': 'high_sbp_any_gt_120', 'source': 'Appendix', 'known_from': 'during stay',
     'definition': 'max plausible systolic BP > 120 mmHg',
     'rationale': 'Appendix threshold; very common in an ED population'},
    {'pattern': 'high_dbp_any_gt_80', 'source': 'Appendix', 'known_from': 'during stay',
     'definition': 'max plausible diastolic BP > 80 mmHg',
     'rationale': 'Appendix threshold; very common in an ED population'},
    {'pattern': 'non_sinus_rhythm', 'source': 'Appendix', 'known_from': 'during stay',
     'definition': 'a recorded rhythm different from sinus rhythm',
     'rationale': 'Appendix threshold; recorded on less than 1% of events'},
    {'pattern': 'repeated_vitals', 'source': 'process', 'known_from': 'during stay',
     'definition': 'at least 2 Vital sign check events',
     'rationale': 'monitoring intensity: a process property, not a clinical one'},
    {'pattern': 'repeated_medication', 'source': 'process', 'known_from': 'during stay',
     'definition': 'at least 3 medication events (reconciliation + dispensations)',
     'rationale': 'pharmacological workload of the stay'},
    {'pattern': 'has_implausible_value', 'source': 'process', 'known_from': 'during stay',
     'definition': 'at least one physiologically implausible recording',
     'rationale': 'data-quality property, kept visible instead of hidden (NB01-C11)'},
    {'pattern': 'acuity_missing', 'source': 'process', 'known_from': 'triage',
     'definition': 'no acuity recorded for the stay',
     'rationale': 'the 56 stays without acuity are kept and made explicit, never dropped'},
    {'pattern': 'multiple_diagnoses', 'source': 'scenario', 'known_from': 'DISCHARGE',
     'definition': 'at least 2 distinct diagnosis codes',
     'rationale': 'clinical complexity - but recorded only at discharge: outcome side'},
    {'pattern': 'admitted_or_transfer', 'source': 'scenario', 'known_from': 'DISCHARGE',
     'definition': 'disposition in {ADMITTED, TRANSFER}',
     'rationale': 'heavier clinical pathway - but it IS the outcome: never a predictor'},
    {'pattern': 'interrupted_pathway', 'source': 'scenario', 'known_from': 'DISCHARGE',
     'definition': 'disposition in {LWBS, ELOPED, AMA, EXPIRED, OTHER}',
     'rationale': 'clinically interrupted stay - outcome side'},
    {'pattern': 'long_stay_p75', 'source': 'process', 'known_from': 'DISCHARGE',
     'definition': 'lead time >= p75 of the distribution',
     'rationale': 'derived from the duration: it is the NB05 target, never a predictor'},
])

# The six patterns of the CORE dictionary, approved for the interpretable
# analysis: all known by triage time, so NB05 can reuse them without leakage
CORE_PATTERNS = ['acuity_1_3', 'acuity_1_2', 'ambulance_arrival',
                 'pain_any_gt_0', 'tachycardia_any_gt_100', 'vitals_before_triage']
pattern_dictionary['in_core_phi'] = pattern_dictionary['pattern'].isin(CORE_PATTERNS)

print('Patterns in the dictionary: {} (core: {})'.format(len(pattern_dictionary), len(CORE_PATTERNS)))
pattern_dictionary

In [ ]:
# [NB04-C05] THE BINARY MAPPING FUNCTION phi
# What: implement phi, the function that maps each stay to a binary vector
#       over the properties of the dictionary.
# Why: this is the method the additional question asks for. Each 1 means the
#      property holds for that stay, each 0 that it does not. The clinical
#      bits reuse the *_abnormal flags of NB01, which are already computed on
#      plausible values only, so an implausible recording can never create a
#      false clinical pattern.
# Expected: a 1,820 x 20 binary matrix, one row per stay.

# Group the raw log by stay: every clinical pattern is an aggregation per case
grouped = raw_log.groupby('case:concept:name')

# The activity sequence of each stay, needed by the process patterns
sequences = raw_log.groupby('case:concept:name')['concept:name'].agg(tuple)

# Helper: does a Vital sign check occur before the Triage event of this stay?
def has_vitals_before_triage(sequence):
    if 'Triage in the ED' not in sequence:
        return False
    position_of_triage = sequence.index('Triage in the ED')
    return 'Vital sign check' in sequence[:position_of_triage]

# Build the binary matrix, one column per property
phi = pd.DataFrame(index=case_table.index)

# --- clinical properties (Appendix thresholds, on plausible values only)
phi['acuity_1_3'] = case_table['acuity'].isin([1.0, 2.0, 3.0])
phi['acuity_1_2'] = case_table['acuity'].isin([1.0, 2.0])
phi['pain_any_gt_0'] = grouped['pain_abnormal'].max().reindex(case_table.index).fillna(False)
phi['tachycardia_any_gt_100'] = grouped['heartrate_abnormal'].max().reindex(case_table.index).fillna(False)
phi['fever_max_ge_100F'] = grouped['temperature_abnormal'].max().reindex(case_table.index).fillna(False)
phi['tachypnea_any_gt_20'] = grouped['resprate_abnormal'].max().reindex(case_table.index).fillna(False)
phi['hypoxemia_any_lt_90'] = grouped['o2sat_abnormal'].max().reindex(case_table.index).fillna(False)
phi['high_sbp_any_gt_120'] = grouped['sbp_abnormal'].max().reindex(case_table.index).fillna(False)
phi['high_dbp_any_gt_80'] = grouped['dbp_abnormal'].max().reindex(case_table.index).fillna(False)
phi['non_sinus_rhythm'] = grouped['rhythm_abnormal'].max().reindex(case_table.index).fillna(False)

# --- context properties (case attributes)
phi['ambulance_arrival'] = case_table['arrival_transport'].eq('AMBULANCE')

# --- process properties
phi['vitals_before_triage'] = sequences.apply(has_vitals_before_triage).reindex(case_table.index).fillna(False)
phi['repeated_vitals'] = case_table['vital_sign_check_count'].ge(2)
phi['repeated_medication'] = (case_table['medicine_reconciliation_count'] + case_table['medicine_dispensations_count']).ge(3)
implausible_columns = [c for c in raw_log.columns if c.endswith('_implausible')]
phi['has_implausible_value'] = grouped[implausible_columns].max().max(axis=1).reindex(case_table.index).fillna(False)
phi['acuity_missing'] = case_table['acuity'].isna()

# --- outcome properties (known only at discharge: description only, never prediction)
phi['multiple_diagnoses'] = case_table['diagnosis_count'].ge(2)
phi['admitted_or_transfer'] = case_table['disposition'].isin(['ADMITTED', 'TRANSFER'])
phi['interrupted_pathway'] = case_table['interrupted_pathway']
phi['long_stay_p75'] = case_table['lead_time_min'].ge(case_table['lead_time_min'].quantile(0.75))

# Everything must be a clean boolean: no NaN can survive into a binary vector
phi = phi.astype(bool)
print('phi matrix: {} stays x {} properties'.format(phi.shape[0], phi.shape[1]))

# Support of each property, with the moment it becomes known
support = pd.DataFrame({'stays': phi.sum(), 'support_pct': (phi.mean() * 100).round(1)})
support = support.merge(pattern_dictionary.set_index('pattern')[['source', 'known_from', 'in_core_phi']],
                        left_index=True, right_index=True)
support = support.sort_values('support_pct', ascending=False)

# Save the dictionary with the measured support for the report
dictionary_export = pattern_dictionary.merge(support[['stays', 'support_pct']],
                                             left_on='pattern', right_index=True, how='left')
dictionary_export.to_csv('pattern_dictionary.csv', index=False)
print('Saved pattern_dictionary.csv')

support

# Interpretation: the support column is already a finding. acuity_1_3 holds
# for 88.7% of stays: faithful to the Appendix, but as a splitter it is
# almost constant - which is exactly why acuity_1_2 (40.6%) was added as a
# declared refinement. At the other end, hypoxemia holds for 1.3% of stays:
# a real clinical signal, but too rare to form a group of its own.

In [ ]:
# [NB04-C06] CONTEXT VARIANTS: THE BINARY SIGNATURE OF EACH STAY
# What: build the context variant of each stay as the string of its core phi
#       bits, in the same format as the example of the case study.
# Why: two stays belong to the same context variant when they satisfy exactly
#      the same properties - an equivalence class over properties instead of
#      over activity sequences (paper CR-219, "Variants of Variants"). This
#      is the answer to the additional question.
# Expected: far fewer groups than the 896 sequential variants of NB02.

# The signature is the concatenation of the core bits, in dictionary order
def build_signature(row):
    bits = []
    for pattern in CORE_PATTERNS:
        bits.append(str(int(row[pattern])))
    return ''.join(bits)
phi['context_signature'] = phi.apply(build_signature, axis=1)

# Note on the two acuity bits: acuity_1_2 implies acuity_1_3, so they are
# NOT independent. Together they encode an ordinal in "thermometer" form:
#   11 -> very urgent (acuity 1-2) | 01 -> urgent (acuity 3) | 00 -> non-urgent (4-5)
# The combination 10 is impossible by construction, and we verify it here.
impossible = phi[phi['acuity_1_2'] & ~phi['acuity_1_3']]
print('Stays with the impossible acuity combination (must be 0): {}'.format(len(impossible)))

# How many context variants does the core phi produce?
signature_counts = phi['context_signature'].value_counts()
print('\nContext variants (core phi, {} bits): {}'.format(len(CORE_PATTERNS), len(signature_counts)))
print('Largest context variant: {} stays'.format(signature_counts.iloc[0]))
print('Context variants with a single stay: {}'.format((signature_counts == 1).sum()))
print('Compare with the 896 SEQUENTIAL variants of NB02-C13.')

# The table in the format of the case-study example: one row per stay,
# one column per property, plus the variant label
example_table = phi[CORE_PATTERNS].astype(int).copy()
example_table['CONTEXT_VARIANT'] = 'V' + phi['context_signature']
print('\nThe encoding, in the format of the case-study example:')
example_table.head(8)

In [ ]:
# [NB04-C07] WHY THE PATTERN SET MUST STAY SMALL
# What: measure how many context variants each pattern set produces, from
#       the core six up to the full twenty.
# Why: this is the trade-off at the heart of the method. Adding properties
#      adds information but multiplies the equivalence classes: past a point
#      the encoding recreates the very fragmentation it was meant to cure.
#      The number must be measured, not guessed.
# Expected: the full dictionary produces MORE groups than the 896 sequential
#      variants we started from - the method defeating itself.

# Helper: how many distinct signatures does a set of patterns produce?
def count_signatures(columns):
    signatures = phi[columns].astype(int).astype(str).agg(''.join, axis=1)
    return signatures.nunique(), signatures.value_counts()

# Pattern sets of growing size
clinical_patterns = ['acuity_1_3', 'acuity_1_2', 'pain_any_gt_0', 'tachycardia_any_gt_100',
                     'fever_max_ge_100F', 'tachypnea_any_gt_20', 'hypoxemia_any_lt_90',
                     'high_sbp_any_gt_120', 'high_dbp_any_gt_80', 'non_sinus_rhythm']
process_patterns = ['vitals_before_triage', 'repeated_vitals', 'repeated_medication',
                    'has_implausible_value', 'acuity_missing']
outcome_patterns = ['multiple_diagnoses', 'admitted_or_transfer', 'interrupted_pathway', 'long_stay_p75']
all_patterns = clinical_patterns + ['ambulance_arrival'] + process_patterns + outcome_patterns

fragmentation_rows = []
for set_name, columns in [('core phi (6)', CORE_PATTERNS),
                          ('clinical (10)', clinical_patterns),
                          ('clinical + context (11)', clinical_patterns + ['ambulance_arrival']),
                          ('+ process (16)', clinical_patterns + ['ambulance_arrival'] + process_patterns),
                          ('full dictionary (20)', all_patterns)]:
    n_signatures, counts = count_signatures(columns)
    fragmentation_rows.append({'pattern_set': set_name, 'patterns': len(columns),
                               'context_variants': n_signatures,
                               'largest_group': int(counts.iloc[0]),
                               'single_stay_groups': int((counts == 1).sum())})
fragmentation = pd.DataFrame(fragmentation_rows)
fragmentation.to_csv('phi_fragmentation.csv', index=False)
print('Saved phi_fragmentation.csv\n')
print(fragmentation.to_string(index=False))

# Plot the fragmentation curve against the sequential-variant baseline
plt.figure(figsize=(9, 5))
plt.plot(fragmentation['patterns'], fragmentation['context_variants'], marker='o', color='#1f4e79',
         label='context variants produced by phi')
plt.axhline(896, color='#c55a11', linestyle='--', label='896 sequential variants (NB02)')
plt.xlabel('Number of properties in phi')
plt.ylabel('Distinct context variants')
plt.title('The parsimony trade-off: more properties, more fragmentation')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

# Save the figure for the report, then show it
plt.savefig('figures/NB04_fragmentation_curve.png', dpi=200)
plt.show()

# Interpretation: the full dictionary produces more equivalence classes than
# the sequential variants it was supposed to replace - the encoding would
# rebuild the same spaghetti in another notation. This is why the core phi is
# deliberately small: parsimony is not an aesthetic preference here, it is the
# condition for the method to work. When more properties must be kept, the
# answer is not more signatures but CLUSTERING over them, done below.

In [ ]:
# [NB04-C08] COMPARING THE CONTEXT VARIANTS
# What: compare the largest context variants on the KPIs of NB02: stay
#       duration, clinical workload and outcome.
# Why: a grouping is only useful if the groups behave differently. Groups
#      with fewer than 20 stays are reported but not tested, for the same
#      reason the head threshold of NB02 was set at 20: a p90 on 3
#      observations is not a statistic.
# Expected: context variants with clearly different durations.

# Attach the KPIs of the case table to the signatures
context = phi[['context_signature']].join(
    case_table[['lead_time_min', 'disposition', 'acuity', 'event_count']])

# Statistics per context variant (all of them: nothing is filtered away)
def p90(values):
    return values.quantile(0.90)
by_variant = context.groupby('context_signature')['lead_time_min'].agg(['size', 'median', p90])
by_variant.columns = ['stays', 'median_min', 'p90_min']
by_variant['admitted_pct'] = context.groupby('context_signature')['disposition'].apply(
    lambda x: round(x.isin(['ADMITTED', 'TRANSFER']).mean() * 100, 1))
by_variant = by_variant.sort_values('stays', ascending=False)

# Save the full table for the report
by_variant.to_csv('context_variants.csv')
print('Saved context_variants.csv ({} context variants)'.format(len(by_variant)))

# Show the groups large enough to be read (>= 20 stays, as in NB02)
readable = by_variant[by_variant['stays'] >= 20]
print('\nContext variants with at least 20 stays: {} (covering {} of {} stays, {:.1f}%)'.format(
    len(readable), int(readable['stays'].sum()), len(case_table),
    readable['stays'].sum() / len(case_table) * 100))
print('\nBits, in order: {}'.format(' | '.join(CORE_PATTERNS)))
readable

In [ ]:
# [NB04-C09] DO THE CONTEXT VARIANTS DIFFER SIGNIFICANTLY?
# What: test whether the readable context variants differ in stay duration.
# Why: the same discipline used for the sequential segments in NB03-C12: a
#      grouping that does not separate behaviour is bookkeeping, not
#      knowledge. Kruskal-Wallis because durations are not normal (NB01-C14).
# Expected: a significant difference would validate the encoding.

# Only the groups with at least 20 stays enter the test (declared choice):
# the others remain in the data and in the exported table
readable_signatures = readable.index.tolist()
tested = context[context['context_signature'].isin(readable_signatures)]
print('Stays in the tested groups: {} of {} ({:.1f}%)'.format(
    len(tested), len(context), len(tested) / len(context) * 100))
print('Stays in the smaller groups (kept, not tested): {}'.format(len(context) - len(tested)))

# Kruskal-Wallis across the readable context variants
groups = []
for signature, group in tested.groupby('context_signature'):
    groups.append(group['lead_time_min'].dropna().values)
statistic, p_value = kruskal(*groups)
print('\nKruskal-Wallis H = {:.2f}, p-value = {:.6f}'.format(statistic, p_value))
if p_value < 0.05:
    print('The context variants differ significantly in stay duration.')
else:
    print('No significant duration difference across context variants.')

# The two extreme groups, to make the difference concrete
slowest = readable['median_min'].idxmax()
fastest = readable['median_min'].idxmin()
print('\nSlowest context variant V{}: {} stays, median {:.0f} min, {}% admitted'.format(
    slowest, int(readable.loc[slowest, 'stays']), readable.loc[slowest, 'median_min'],
    readable.loc[slowest, 'admitted_pct']))
print('Fastest context variant V{}: {} stays, median {:.0f} min, {}% admitted'.format(
    fastest, int(readable.loc[fastest, 'stays']), readable.loc[fastest, 'median_min'],
    readable.loc[fastest, 'admitted_pct']))
print('\nBits, in order: {}'.format(' | '.join(CORE_PATTERNS)))

In [ ]:
# [NB04-C10] CLUSTERING THE SIGNATURES (THE SCALABLE PATH)
# What: when the properties are many and the signatures fragment, group the
#       stays by clustering their binary vectors instead of reading each
#       signature: agglomerative clustering on the Jaccard distance, with the
#       number of clusters chosen by silhouette.
# Why: this is the answer to the fragmentation measured in NB04-C07, and the
#      method of Case8Clustering adapted to binary data (Jaccard is the right
#      distance for binary vectors; Euclidean would treat 0s as information).
#      The full dictionary is used here, minus the outcome properties, which
#      would group stays by their result instead of their context.
# Expected: a handful of clusters covering all 1,820 stays.

# Features for the clustering: everything except the outcome-side properties
clustering_patterns = clinical_patterns + ['ambulance_arrival'] + process_patterns
X = phi[clustering_patterns].astype(int)
print('Clustering on {} properties x {} stays (outcome properties excluded)'.format(
    X.shape[1], X.shape[0]))

# Jaccard distance between every pair of stays
distances = pairwise_distances(X.values, metric='jaccard')

# Try a few numbers of clusters and keep the silhouette of each
silhouette_rows = []
for k in range(2, 9):
    labels = AgglomerativeClustering(n_clusters=k, metric='precomputed',
                                     linkage='average').fit_predict(distances)
    score = silhouette_score(distances, labels, metric='precomputed')
    silhouette_rows.append({'clusters': k, 'silhouette': round(score, 3)})
    print('k = {} -> silhouette {:.3f}'.format(k, score))
silhouette_table = pd.DataFrame(silhouette_rows)

# Keep the k with the best silhouette
best_k = int(silhouette_table.loc[silhouette_table['silhouette'].idxmax(), 'clusters'])
print('\nBest number of clusters by silhouette: {} (best silhouette {:.3f})'.format(
    best_k, silhouette_table['silhouette'].max()))
labels = AgglomerativeClustering(n_clusters=best_k, metric='precomputed',
                                 linkage='average').fit_predict(distances)
phi['context_cluster'] = labels.astype(str)

# Verify: every stay belongs to exactly one cluster, none is lost
print('Stays assigned to a cluster: {} of {}'.format(phi['context_cluster'].notna().sum(), len(phi)))

# Is the clustering actually usable? Check whether it is degenerate, i.e. one
# giant cluster plus a few singletons. This is a NEGATIVE RESULT worth
# reporting, not a failure to hide.
cluster_sizes = phi['context_cluster'].value_counts()
print('\nCluster sizes: {}'.format(dict(cluster_sizes)))
largest_share = cluster_sizes.iloc[0] / len(phi)
print('Largest cluster holds {:.1f}% of the stays; silhouette peaks at {:.3f}.'.format(
    largest_share * 100, silhouette_table['silhouette'].max()))
if largest_share > 0.9:
    print('DEGENERATE: the binary signatures do NOT form natural clusters on this population.')

# Interpretation: the clustering is degenerate - one cluster holds almost all
# the stays and the silhouette is low at every k. This is not a bug, it is a
# finding: in an ED population almost every patient shares the common
# properties (high SBP 80%, repeated vitals 73%, some pain 60%), so the
# Jaccard distance sees everyone as similar and no natural grouping emerges.
# The consequence is methodological and goes into the report: for THIS data
# the interpretable context variants of NB04-C06 (44 groups, 25 readable
# covering 90.9%) are the right tool, and data-driven clustering - the path
# used in Case8Clustering on continuous features - does not add structure on
# binary signatures. A negative result that reinforces the parsimony
# argument of NB04-C07.

In [ ]:
# [NB04-C11] WHAT EACH CLUSTER IS MADE OF
# What: describe each cluster by its size, its KPIs and the properties that
#       characterise it (the share of stays in the cluster holding each bit).
# Why: a cluster label is useless until it can be named in clinical terms.
#      The profile table turns "cluster 2" into a description an ED manager
#      can recognise.
# Expected: clusters with distinct clinical profiles and distinct durations.

# KPIs per cluster
cluster_context = phi[['context_cluster']].join(
    case_table[['lead_time_min', 'disposition', 'acuity', 'event_count']])
cluster_summary = cluster_context.groupby('context_cluster')['lead_time_min'].agg(['size', 'median', p90])
cluster_summary.columns = ['stays', 'median_min', 'p90_min']
cluster_summary['admitted_pct'] = cluster_context.groupby('context_cluster')['disposition'].apply(
    lambda x: round(x.isin(['ADMITTED', 'TRANSFER']).mean() * 100, 1))
print('KPIs per cluster:')
print(cluster_summary.round(1))

# Profile of each cluster: share of stays holding each property
profile = phi.groupby('context_cluster')[clustering_patterns].mean().round(2)
print('\nProfile of each cluster (share of stays holding each property):')
print(profile.T)

# Save both tables for the report
cluster_summary.to_csv('context_cluster_summary.csv')
profile.T.to_csv('context_cluster_profile.csv')
print('\nSaved context_cluster_summary.csv and context_cluster_profile.csv')

# Do the clusters differ in duration?
cluster_groups = []
for cluster_name, group in cluster_context.groupby('context_cluster'):
    cluster_groups.append(group['lead_time_min'].dropna().values)
statistic, p_value = kruskal(*cluster_groups)
print('\nKruskal-Wallis across clusters: H = {:.2f}, p-value = {:.6f}'.format(statistic, p_value))
if p_value < 0.05:
    print('The context clusters differ significantly in stay duration.')
else:
    print('No significant duration difference across clusters.')

# Boxplot of the duration per cluster (outliers hidden from the PLOT only)
plt.figure(figsize=(9, 5))
sns.boxplot(data=cluster_context, x='context_cluster',
            y=cluster_context['lead_time_min'] / 60, showfliers=False, color='#9dc3e6')
plt.xlabel('Context cluster')
plt.ylabel('Stay duration (hours)')
plt.title('Stay duration by context cluster (k = {})'.format(best_k))
plt.tight_layout()

# Save the figure for the report, then show it
plt.savefig('figures/NB04_duration_by_cluster.png', dpi=200)
plt.show()

In [ ]:
# [NB04-C12] A PROCESS MODEL FOR EACH CONTEXT GROUP
# What: discover a Petri net on the sub-log of each context cluster and
#       compare the models.
# Why: this closes the loop of the method (Case8ContextBased): the context
#      does not only change the KPIs, it can change the PROCESS. Comparing
#      the models per context group shows whether patients with different
#      profiles are treated through structurally different paths.
# Expected: clusters with different model complexity.

# Discover and evaluate a model per cluster (sub-logs are views: nothing filtered)
model_rows = []
for cluster_name in sorted(phi['context_cluster'].unique()):
    cluster_stays = phi[phi['context_cluster'] == cluster_name].index
    sub_log = abstract_log[abstract_log['case:concept:name'].isin(cluster_stays)]
    net, im, fm = pm4py.discover_petri_net_inductive(sub_log, noise_threshold=0.2)
    fitness_result = pm4py.fitness_token_based_replay(sub_log, net, im, fm)
    fitness = fitness_result.get('average_trace_fitness', fitness_result.get('log_fitness', np.nan))
    precision = pm4py.precision_token_based_replay(sub_log, net, im, fm)
    activities = [t.label for t in net.transitions if t.label is not None]
    model_rows.append({'cluster': cluster_name, 'stays': len(cluster_stays),
                       'fitness': round(fitness, 3), 'precision': round(precision, 3),
                       'activities': len(activities), 'places': len(net.places), 'arcs': len(net.arcs)})
    print('cluster {} ({:4d} stays) -> fitness {:.3f} | precision {:.3f} | activities {} | places {} | arcs {}'.format(
        cluster_name, len(cluster_stays), fitness, precision, len(activities), len(net.places), len(net.arcs)))

cluster_models = pd.DataFrame(model_rows)
cluster_models.to_csv('context_cluster_models.csv', index=False)
print('\nSaved context_cluster_models.csv')

cluster_models

# Interpretation: read this table together with the profile of NB04-C11. A
# cluster whose model has more places and arcs is a group whose patients
# follow more varied paths - a candidate for standardisation - while a
# compact, precise model marks a group already following a routine.

In [ ]:
# [NB04-C13] FROM PATTERNS TO IMPROVEMENT OPTIONS
# What: turn each pattern into the insight it produced and the improvement
#       option it suggests, with the KPI that would verify it.
# Why: the additional question asks explicitly "How can this method help to
#      identify improvements? Suggest some options". This table is the
#      answer, and it is built only from evidence measured in this project.
# Expected: a table for the report, with the honest limits attached.

# Measure the duration gap each core pattern is associated with: the
# difference in median stay duration between stays holding it and the others
evidence_rows = []
for pattern in CORE_PATTERNS:
    with_pattern = case_table.loc[phi[phi[pattern]].index, 'lead_time_min']
    without_pattern = case_table.loc[phi[~phi[pattern]].index, 'lead_time_min']
    statistic, p_value = mannwhitneyu(with_pattern, without_pattern, alternative='two-sided')
    evidence_rows.append({'pattern': pattern,
                          'stays_with': len(with_pattern),
                          'median_with_min': round(with_pattern.median(), 1),
                          'median_without_min': round(without_pattern.median(), 1),
                          'gap_min': round(with_pattern.median() - without_pattern.median(), 1),
                          'mann_whitney_p': round(p_value, 4)})
pattern_evidence = pd.DataFrame(evidence_rows).sort_values('gap_min', ascending=False)
pattern_evidence.to_csv('pattern_evidence.csv', index=False)
print('Saved pattern_evidence.csv')
print('\nDuration gap associated with each core pattern (NOT a causal effect):')
print(pattern_evidence.to_string(index=False))

# Interpretation: these gaps are associations, never causes. A stay is not
# long BECAUSE the patient arrived by ambulance: both are consequences of a
# clinical condition the log does not record. The improvement table of the
# report states this limit next to every option, and no option is proposed
# unless the evidence converges (a duration gap AND a group AND a deviation).

In [ ]:
# [NB04-C14] EXPORTS AND NO-DATA-LOSS CHECK
# What: save the phi matrix and verify that no stay was lost.
# Why: project rule - nothing is ever dropped; every notebook ends by proving
#      it. Every stay has a signature and a cluster, including the 56 without
#      acuity (which carry the acuity_missing bit instead of disappearing).
# Expected: 1,820 stays with a signature and a cluster.

# Verify: every stay has a signature and a cluster
print('Stays with a context signature: {} of {}'.format(phi['context_signature'].notna().sum(), len(case_table)))
print('Stays with a context cluster: {} of {}'.format(phi['context_cluster'].notna().sum(), len(case_table)))
print('Stays without acuity, kept and flagged: {}'.format(int(phi['acuity_missing'].sum())))

# Save the full phi matrix (binary vectors + signature + cluster) for NB05
phi.to_csv('phi_matrix.csv')
print('\nSaved phi_matrix.csv ({} stays x {} columns)'.format(phi.shape[0], phi.shape[1]))

# Recap of the artifacts
print('\nNB04 outputs: pattern_dictionary.csv, phi_matrix.csv, phi_fragmentation.csv,')
print('context_variants.csv, context_cluster_summary.csv, context_cluster_profile.csv,')
print('context_cluster_models.csv, pattern_evidence.csv, 2 figures in figures/')
print('\nNo stay, variant or event was removed. The properties known only at')
print('discharge (disposition, diagnosis, lead time) are marked in the')
print('dictionary and are excluded from the predictive features of NB05.')